In [1]:
from pathlib import Path

import pymupdf
import pandas as pd

pdf_path = Path("../data/2025 한국 반려동물 보고서.pdf")

if not pdf_path.exists():
    raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {pdf_path.resolve()}")

with pymupdf.open(pdf_path) as pdf:
    pages = [
        {
            "page": page_number,
            "text": page.get_text("text").strip(),
        }
        for page_number, page in enumerate(pdf, start=1)
    ]

pdf_df = pd.DataFrame(pages)
pdf_df["text_length"] = pdf_df["text"].str.len()

print(f"파일: {pdf_path.resolve()}")
print(f"페이지 수: {len(pdf_df)}")
pdf_df.head()

파일: C:\Users\rhksa\Desktop\mle-01-p1-team2\data\2025 한국 반려동물 보고서.pdf
페이지 수: 112


,page,text,text_length
0,1,반려동물 건강 웰니스와 비만 관리\n2025 한국 반려동물 보고서\n황원경 | 김남...,59
1,2,"2025 한국 반려동물 보고서\n국내 591만 가구, 1,546만 명이 반려동물 양...",1198
2,3,Infographic\n펫로스경험유무\n54.7\n45.3\n●있다●없다\n24.3...,1119
3,4,2025 한국반려동물보고서\n“한국반려가구에게반려동물은가족”\n어떻게알수있을까요 \...,1108
4,5,“최근부쩍관심이높아진반려동물건강웰니스”\n반려가구는무엇을하고있나요 \n반려가구의반려...,1108


In [6]:
# PDF 전체 및 페이지별 글자 수 확인
page_char_stats = pdf_df["text"].str.len()

print(f"전체 글자 수: {page_char_stats.sum():,}")
print(f"평균 페이지 글자 수: {page_char_stats.mean():,.0f}")
print(f"중앙값 페이지 글자 수: {page_char_stats.median():,.0f}")
print(f"최소 페이지 글자 수: {page_char_stats.min():,}")
print(f"최대 페이지 글자 수: {page_char_stats.max():,}")

pdf_df[["page", "text_length"]].describe().round(0)

전체 글자 수: 102,479
평균 페이지 글자 수: 915
중앙값 페이지 글자 수: 945
최소 페이지 글자 수: 0
최대 페이지 글자 수: 1,584


,page,text_length
count,112.0,112.0
mean,56.0,915.0
std,32.0,356.0
min,1.0,0.0
25%,29.0,758.0
50%,56.0,945.0
75%,84.0,1157.0
max,112.0,1584.0


In [3]:
# 청크 크기 후보별 예상 청크 수 비교
chunk_sizes = [500, 800, 1000, 1500]
overlap = 100
non_empty_char_count = page_char_stats[page_char_stats > 0].sum()

chunk_plan = pd.DataFrame(
    {
        "chunk_size": chunk_sizes,
        "overlap": overlap,
        "estimated_chunks": [
            (non_empty_char_count + (size - overlap) - 1) // (size - overlap)
            for size in chunk_sizes
        ],
    }
)

chunk_plan["overlap_ratio"] = (
    chunk_plan["overlap"] / chunk_plan["chunk_size"]
).round(2)
chunk_plan

,chunk_size,overlap,estimated_chunks,overlap_ratio
0,500,100,257,0.20
1,800,100,147,0.12
2,1000,100,114,0.10
3,1500,100,74,0.07


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re

chunk_size = 1000
overlap = 200
control_char_pattern = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=overlap,
    separators=["\n\n", "\n", " ", ""],
    length_function=len,
)

chunks = []
for page_number, page_text in zip(pdf_df["page"], pdf_df["text"]):
    if not page_text:
        continue

    cleaned_page_text = control_char_pattern.sub(" ", page_text)
    page_chunks = text_splitter.split_text(cleaned_page_text)
    chunks.extend(
        {
            "chunk_id": f"page-{page_number}-chunk-{chunk_index}",
            "page": page_number,
            "text": chunk_text,
            "text_length": len(chunk_text),
        }
        for chunk_index, chunk_text in enumerate(page_chunks, start=1)
    )

chunks_df = pd.DataFrame(chunks)

print(f"청크 수: {len(chunks_df)}")
print(f"청크 크기: 최대 {chunks_df['text_length'].max():,}자")
print(f"청크 오버랩: {overlap}자")
chunks_df.head()

청크 수: 158
청크 크기: 최대 1,000자
청크 오버랩: 200자


,chunk_id,page,text,text_length
0,page-1-chunk-1,1,반려동물 건강 웰니스와 비만 관리\n2025 한국 반려동물 보고서\n황원경 | 김남...,59
1,page-2-chunk-1,2,"2025 한국 반려동물 보고서\n국내 591만 가구, 1,546만 명이 반려동물 양...",999
2,page-2-chunk-2,2,13.5\n16.1\n12.6\n14.2\n전체\n19.1\n22.5\n31.7\n...,393
3,page-3-chunk-1,3,Infographic\n펫로스경험유무\n54.7\n45.3\n●있다●없다\n24.3...,995
4,page-3-chunk-2,3,(단위: %)\n(단위: %)\n저축·자금운용유무\n26.6\n73.4\n●있다●없...,319


In [8]:
import hashlib

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

project_dir = pdf_path.resolve().parent.parent
chroma_dir = project_dir / "data" / "chroma_db"
collection_name = "pet_analysis"
embedding_model_name = "jhgan/ko-sroberta-multitask"

embedding_model = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    encode_kwargs={"normalize_embeddings": True},
)

vector_db = Chroma(
    collection_name=collection_name,
    embedding_function=embedding_model,
    persist_directory=str(chroma_dir),
)

source_name = pdf_path.name
source_id = hashlib.sha256(str(pdf_path.resolve()).encode("utf-8")).hexdigest()[:16]

metadatas = [
    {
        "source": source_name,
        "source_path": str(pdf_path.resolve()),
        "page": int(row.page),
        "chunk_index": int(row.chunk_id.rsplit("-", 1)[-1]) - 1,
        "chunk_size": chunk_size,
        "chunk_overlap": overlap,
        "collection": collection_name,
    }
    for row in chunks_df.itertuples(index=False)
]
ids = [
    f"{source_id}-p{metadata['page']:04d}-c{metadata['chunk_index']:04d}"
    for metadata in metadatas
]

existing = vector_db.get(where={"source": source_name}, include=[])
if existing["ids"]:
    vector_db.delete(ids=existing["ids"])

vector_db.add_texts(
    texts=chunks_df["text"].tolist(),
    metadatas=metadatas,
    ids=ids,
)

print(f"적재 완료: {len(ids):,}개")
print(f"컬렉션 전체 문서 수: {vector_db._collection.count():,}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2978.34it/s]


적재 완료: 158개
컬렉션 전체 문서 수: 158


In [9]:
query = "반려동물 양육 가구의 주요 관심사"
results = vector_db.similarity_search(query, k=3)

for index, result in enumerate(results, start=1):
    page = result.metadata.get("page")
    print(f"[{index}] 페이지: {page}")
    print(result.page_content[:300].replace("\n", " ") + "\n")

[1] 페이지: 8
Contents ____Ⅰ 한국 반려동물 현황 01 | 한국 반려동물 양육 현황	 2 02 | 향후 양육 희망 반려동물	 6 03 | 선호 품종과 입양처	 8 04 | 관련 법·제도 강화 의견	 12 05 | 펫티켓 성숙도	 16 Key Findings	 22 ____Ⅱ 반려동물의 생활 웰니스 01 | 반려동물 웰니스 인식	 24 02 | 반려동물의 영양 관리	 26 03 | 반려동물의 운동과 놀이	 30 04 | ‘나홀로 집에’ 반려동물 케어	 32 05 | 반려동물과의 여가활동	 34 06 | 반려동물을 위한 건강검진	 38 K

[2] 페이지: 52
 ‘교육’	 위→ 위), ‘외출’	 위→ 위), ‘자금’	 위→ 위출’(22.3%), 금융상품·양육비 용등‘자금’(21.9%) 관련분야순으로조사됐다      년조사와비교해‘건강관리’(55.0%)와‘양육’(38.8%)에대한 반려인의높은관심은변함없이지속됐고

[3] 페이지: 42
 반려견가구와반려묘가구의경우모두    년대비나홀로집에있는 비중을소폭이나마줄였다	    년대비‘반려견’  0.9%p, ‘반려묘’      Q 

